# Summarizing Text with Large Language Models

In high-volume domains like e-commerce, customer support, and content intelligence, processing extensive user-generated text is a core operational challenge. 

Large Language Models (LLMs) provide an automated mechanism to compress, synthesize, and filter textual data. This notebook covers key patterns for summarization:

1. **Baseline Summarization with Output Constraints:** Governing length via word, sentence, and character boundaries.
2. **Department-Specific Targeted Summarization:** Steering focus toward distinct operational lenses (e.g., Shipping vs. Pricing).
3. **Summarization vs. Information Extraction:** Eliminating irrelevant contextual noise when strict entity/event retrieval is needed.
4. **Batch Processing Workflows:** Automating multi-document pipelines for executive dashboards and catalog auditing.

## 1. Environment Setup & Configuration

Configure credentials and initialize the client.
- Supports standard OpenAI credentials (`OPENAI_API_KEY`).
- Fully compatible with Google Gemini using its OpenAI-compatible endpoint (`GEMINI_API_KEY` pointed to `https://generativelanguage.googleapis.com/v1beta/openai/`).

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

# 1. Load API key from .env file
_ = load_dotenv(find_dotenv())

# 2. Initialize the OpenAI client pointing to the working Gemini endpoint
client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

### Helper Function: `get_completion`
Encapsulates chat completion requests. `temperature=0` enforces deterministic, reproducible outputs.

In [2]:
# # 3. helper function to make it easier to get completions from the model
# def get_completion(prompt, model="gemini-3.8-flash"):
#     messages = [{"role": "user", "content": prompt}]
#     response = client.chat.completions.create(
#         model=model,
#         messages=messages,
#         temperature=0,  # 0 for deterministic output
#     )
#     return response.choices[0].message.content

# 3. Define prioritized models (from fastest/latest to reliable backups)
from openai import OpenAI, APIError, RateLimitError, InternalServerError, NotFoundError
DEFAULT_MODELS = [
    "gemini-pro-latest",
    "gemini-flash-latest",
    "gemini-flash-lite-latest",
    "gemini-2.0-flash-lite",
    "gemini-3.8-flash",
    "gemini-2.5-flash",
    "gemini-2.0-flash",
    "gemini-1.5-flash",
    "gemini-1.5-pro",
]

def get_completion(prompt, models=DEFAULT_MODELS, max_retries_per_model=2):
    """
    Attempts to get a completion by cascading through a prioritized list of models.
    Handles 429 (Quota/Rate Limits), 503 (High Demand/Unavailable), and 404 (Not Found).
    """
    # If a single model string is passed by mistake, wrap it in a list
    if isinstance(models, str):
        models = [models] + [m for m in DEFAULT_MODELS if m != models]

    messages = [{"role": "user", "content": prompt}]
    last_exception = None

    for model_name in models:
        delay = 1.0  # Initial delay for temporary errors

        for attempt in range(max_retries_per_model):
            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=messages,
                    temperature=0,  # Deterministic output
                )
                return response.choices[0].message.content

            except RateLimitError as e:
                # 429 Quota Exceeded (Per-model limit reached) -> Switch immediately
                print(f"[429 Quota Exceeded] '{model_name}' limit reached. Switching to next model...")
                last_exception = e
                break  # Break inner loop to try next model immediately

            except NotFoundError as e:
                # 404 Model Not Found/Retired -> Switch immediately
                print(f"[404 Not Found] '{model_name}' is unavailable on this API version. Switching...")
                last_exception = e
                break

            except (InternalServerError, APIError) as e:
                # 503 / 500 Temporary Server Overload -> Retry once or switch
                last_exception = e
                status_code = getattr(e, "status_code", None)

                if attempt < max_retries_per_model - 1:
                    print(f"[{status_code or 'Error'}] '{model_name}' busy. Retrying in {delay}s...")
                    time.sleep(delay)
                    delay *= 2  # Exponential backoff
                else:
                    print(f"'{model_name}' failed after {max_retries_per_model} attempts. Switching to next model...")

    # If all models in the list fail:
    raise RuntimeError(f"All configured models failed. Last error: {last_exception}")

###### messages = [{"role": "user", "content": prompt}]: Formats your input prompt into the standard Chat Completion format, indicating that a human user is asking the question.

###### client.chat.completions.create(...): Sends the request to Gemini's server.

###### temperature=0: Sets the degree of randomness. A temperature of 0 ensures the model gives consistent, deterministic, and factual outputs rather than creative variations.

###### return response.choices[0].message.content: Extracts the generated text from the model's response and returns it as a string.

## 2. Source Data: Customer Review Sample

We begin with a multi-faceted e-commerce review evaluating aesthetics, child feedback, perceived size vs. price, and delivery logistics.

In [3]:
prod_review = """
Got this panda plush toy for my daughter's birthday, 
who loves it and takes it everywhere. It's soft and 
super cute, and its face has a friendly look. It's 
a bit small for what I paid though. I think there 
might be other options that are bigger for the 
same price. It arrived a day earlier than expected, 
so I got to play with it myself before I gave it 
to her.
"""

## 3. General Summarization with Length Constraints

### Concept & Implementation
A standard prompt with delimiters (` ``` `) that enforces a strict 30-word limit. This compresses the essential sentiment without losing critical product reception details.

In [4]:
prompt_general = f"""
Your task is to generate a short summary of a product 
review from an ecommerce site. 

Summarize the review below, delimited by triple 
backticks, in at most 30 words. 

Review: ```{prod_review}```
"""

response = get_completion(prompt_general)
print("--- General Summary (<= 30 words) ---")
print(response)
print("\nWord count:", len(response.split()))

[429 Quota Exceeded] 'gemini-pro-latest' limit reached. Switching to next model...
--- General Summary (<= 30 words) ---
The customer's daughter loves this cute, soft panda plush, which arrived early. However, the reviewer felt it was a bit small for the price.

Word count: 24


## 4. Targeted Operational Summarization

In enterprise applications, different teams need completely different takeaways from the exact same raw data:
- **Logistics / Fulfillment:** Care only about dispatch, transit condition, carriers, and delivery timing.
- **Pricing / Merchandising:** Care only about price elasticity, competitive size-to-cost perception, and value satisfaction.

### 4.1 Focus on Shipping & Logistics

In [5]:
prompt_shipping = f"""
Your task is to generate a short summary of a product 
review from an ecommerce site to give feedback to the 
Shipping department. 

Summarize the review below, delimited by triple 
backticks, in at most 30 words, and focusing on any aspects 
that mention shipping and delivery of the product. 

Review: ```{prod_review}```
"""

response_shipping = get_completion(prompt_shipping)
print("--- Shipping-Focused Summary ---")
print(response_shipping)

[429 Quota Exceeded] 'gemini-pro-latest' limit reached. Switching to next model...
--- Shipping-Focused Summary ---
The product arrived a day earlier than expected, providing a timely and positive delivery experience for the customer.


### 4.2 Focus on Pricing & Perceived Value

In [6]:
prompt_pricing = f"""
Your task is to generate a short summary of a product 
review from an ecommerce site to give feedback to the 
pricing department, responsible for determining the 
price of the product.  

Summarize the review below, delimited by triple 
backticks, in at most 30 words, and focusing on any aspects 
that are relevant to the price and perceived value. 

Review: ```{prod_review}```
"""

response_pricing = get_completion(prompt_pricing)
print("--- Pricing-Focused Summary ---")
print(response_pricing)

[429 Quota Exceeded] 'gemini-pro-latest' limit reached. Switching to next model...
--- Pricing-Focused Summary ---
While the customer liked the plush toy's quality, they felt it was too small for the price, noting that larger alternatives are available for the same cost.


## 5. Critical Distinction: "Summarize" vs. "Extract"

### Error Analysis of Focused Summaries
Notice that even when instructed to focus on shipping, the model often retains secondary contextual remarks (e.g., *"plush toy loved by daughter... arrived early"*). While informative, this is not pure operational data.

### The Fix: Direct Information Extraction
Replacing the directive **"Summarize"** with **"Extract"** instructs the model to filter out all non-target commentary, returning only the target topic.

In [ ]:
prompt_extract = f"""
Your task is to extract relevant information from 
a product review from an ecommerce site to give 
feedback to the Shipping department. 

From the review below, delimited by triple backticks, 
extract the information relevant to shipping and 
delivery. Limit to 30 words. 

Review: ```{prod_review}```
"""

response_extract = get_completion(prompt_extract)
print("--- Information Extraction Output ---")
print(response_extract)

## 6. Batch Processing Pipeline: Multi-Document Summarization

In production dashboards, summarization prompts are typically deployed across large queues or streams of incoming documents. 

Below, we process multiple heterogeneous reviews (home goods, personal care, kitchen appliances) in an automated loop.

In [ ]:
# Multi-review corpus across diverse product categories
review_1 = prod_review 

# Review for a standing lamp (logistics damage and support)
review_2 = """
Needed a nice lamp for my bedroom, and this one 
had additional storage and not too high of a price 
point. Got it fast - arrived in 2 days. The string 
to the lamp broke during the transit and the company 
happily sent over a new one. Came within a few days 
as well. It was easy to put together. Then I had a 
missing part, so I contacted their support and they 
very quickly got me the missing piece! Seems to me 
to be a great company that cares about their customers 
and products. 
"""

# Review for an electric toothbrush (ergonomics, battery, and pricing)
review_3 = """
My dental hygienist recommended an electric toothbrush, 
which is why I got this. The battery life seems to be 
pretty impressive so far. After initial charging and 
leaving the charger plugged in for the first week to 
condition the battery, I've unplugged the charger and 
been using it for twice daily brushing for the last 
3 weeks all on the same charge. But the toothbrush head 
is too small. I’ve seen baby toothbrushes bigger than 
this one. I wish the head was bigger with different 
length bristles to get between teeth better because 
this one doesn’t.  Overall if you can get this one 
around the $50 mark, it's a good deal. The manufactuer's 
replacements heads are pretty expensive, but you can 
get generic ones that're more reasonably priced. This 
toothbrush makes me feel like I've been to the dentist 
every day. My teeth feel sparkly clean! 
"""

# Review for a blender (seasonal pricing changes, warranty, and long-term durability)
review_4 = """
So, they still had the 17 piece system on seasonal 
sale for around $49 in the month of November, about 
half off, but for some reason (call it price gouging) 
around the second week of December the prices all went 
up to about anywhere from between $70-$89 for the same 
system. And the 11 piece system went up around $10 or 
so in price also from the earlier sale price of $29. 
So it looks okay, but if you look at the base, the part 
where the blade locks into place doesn’t look as good 
as in previous editions from a few years ago, but I 
plan to be very gentle with it (example, I crush 
very hard items like beans, ice, rice, etc. in the 
blender first then pulverize them in the serving size 
I want in the blender then switch to the whipping 
blade for a finer flour, and use the cross cutting blade 
first when making smoothies, then use the flat blade 
if I need them finer/less pulpy). Special tip when making 
smoothies, finely cut and freeze the fruits and 
vegetables (if using spinach-lightly stew soften the 
spinach then freeze until ready for use-and if making 
sorbet, use a small to medium sized food processor) 
that you plan to use that way you can avoid adding so 
much ice if at all-when making your smoothie. 
After about a year, the motor was making a funny noise. 
I called customer service but the warranty expired 
already, so I had to buy another one. FYI: The overall 
quality has gone done in these types of products, so 
they are kind of counting on brand recognition and 
consumer loyalty to maintain sales. Got it in about 
two days. 
"""

reviews = [review_1, review_2, review_3, review_4]

### Batch Execution Loop
We iterate through all customer reviews, applying a 20-word constraint to standardize reading time for human auditors.

In [ ]:
for idx, review_text in enumerate(reviews, start=1):
    prompt_batch = f"""
    Your task is to generate a short summary of a product 
    review from an ecommerce site. 

    Summarize the review below, delimited by triple 
    backticks in at most 20 words. 

    Review: ```{review_text}```
    """
    
    summary = get_completion(prompt_batch)
    print(f"--- Review {idx} Summary ---")
    print(summary.strip())
    print("-" * 40)

## 7. Strategic Summary & Best Practices

| Technique | Intent | Prompting Strategy |
| :--- | :--- | :--- |
| **Constrained Summarization** | Quick scannability across high text volumes | Enforce hard bounds: `in at most N words` or `in at most N sentences`. |
| **Departmental Steering** | Route insights to relevant business units | State the consumer/stakeholder audience: `...to give feedback to the Shipping / Pricing department`. |
| **Information Extraction** | Isolate target variables with zero conversational filler | Shift task verb: replace `Summarize...` with `Extract information relevant to...`. |
| **Batch Ingestion** | Scale summarization over catalogs & databases | Combine prompt templates with batch loop harnesses for review dashboards and trend analysis. |